# EDA exploration (Pokémon VGC aggregates)

Interactive companion of `python -m python.analytics.eda_report`. The report generates the fixed findings; this notebook is for asking
your own questions on the same aggregate tables. It reuses the report's functions, so numbers cannot differ from `docs/EDA_FINDINGS.md`.

Run from the repository root after the aggregate modules have written `data/aggregates/`. Every table states its population
(regulation, rating band) and sample size; read rates together with them (see `METRICS.md`).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from python.analytics import eda_report as eda
from python.analytics.matchup_stats import opponents_of
from python.analytics.partner_stats import partners_of
from python.analytics.usage_trend import species_trend

AGG = Path(os.environ.get("AGG_DIR", "data/aggregates"))
usage = eda.read_table(AGG, "pokemon_usage_stats")
monthly = eda.read_table(AGG, "pokemon_usage_monthly")
tera_types = eda.read_table(AGG, "pokemon_tera_types")
partner = eda.read_table(AGG, "pokemon_partner_stats")
matchup = eda.read_table(AGG, "pokemon_matchup_stats")
print({name: (None if t is None else len(t)) for name, t in
       [("usage", usage), ("monthly", monthly), ("tera_types", tera_types), ("partner", partner), ("matchup", matchup)]})

## Parameters
Change these three values and re-run the cells below.

In [ ]:
sizes = usage[usage["rating_band"] == "all"].drop_duplicates("regulation").set_index("regulation")["player_battles"]
REGULATION = sizes.idxmax()     # the regulation with the most player-battles; e.g. "gen9vgc2025regi"
SPECIES = "Incineroar"
BAND = "all"                    # all, below_1200, 1200_1399, 1400_1499, 1500_plus
print(REGULATION, SPECIES, BAND, f"({int(sizes[REGULATION]):,} player-battles)")

## 1. How does a species' usage move month by month?
Months flagged `tail` (the format is ending) are shown in the table but left out of the chart.

In [ ]:
t = species_trend(monthly, SPECIES, REGULATION)
display(t)
ok = t[~t["tail"]]
if len(ok):
    ax = ok.plot(x="month", y="usage_%", marker="o", legend=False, title=f"{SPECIES}: usage by month, {REGULATION}")
    ax.set_ylabel("% of teams")
    plt.show()

## 2. What do strong players pick differently?
Usage at 1500+ minus usage below 1200 (percentage points), species with at least 2,000 teams in both bands.

In [ ]:
display(eda.band_shift(usage, REGULATION, n=10))

## 3. Does popularity predict win rate, inside each rating band?
Spearman correlation of usage and win rate relative to the band's own baseline. A value that shrinks inside the bands points to a rating effect.

In [ ]:
display(eda.popularity_by_band(usage[usage["regulation"] == REGULATION]))

## 4. Which Tera types does the species use?

In [ ]:
tt = tera_types[(tera_types["regulation"] == REGULATION) & (tera_types["species"] == SPECIES) & (tera_types["rating_band"] == BAND)]
tt = tt.sort_values("events", ascending=False)
display(tt[["tera_type", "events", "type_share", "wins"]])
if len(tt):
    tt.plot.bar(x="tera_type", y="type_share", legend=False, title=f"Tera types of {SPECIES}, {REGULATION}")
    plt.show()

## 5. Which teammates does it appear with?
`lift` is co-occurrence compared with chance, not proof of synergy.

In [ ]:
display(partners_of(partner, SPECIES, REGULATION, BAND))

## 6. Which opponent teams do teams with it beat or lose to?
Ranked by z-score against what both teams' overall strength predicts (log5); small effects are normal.

In [ ]:
hard, easy = opponents_of(matchup, SPECIES, REGULATION, BAND)
print("hardest"); display(hard)
print("easiest"); display(easy)